# Markov check: do the data contradict a graph?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/markov_check.ipynb)

A graph implies conditional independences: each variable is independent of every earlier variable
in a causal order that is not its parent, given its parents. `andrey.markov_check` tests each one
with a conditional-independence test and asks whether the p-values look uniform, as they do when
every independence holds. A small uniformity p-value rejects the graph. Here the check runs on data
simulated on the ALARM network, whose true graph we know, to show what it catches, what it misses,
and how it can choose among graphs.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.11.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.11.0" pandas

In [2]:
import pandas as pd

import andrey
import andrey.viz
from andrey import GraphStructure
from andrey.data import load_dataset
from andrey.metrics import shd, to_cpdag

ALARM is a published network of 37 variables and 46 arcs. `load_dataset` simulates rows on it from
a linear-Gaussian model. A graph fitted to some rows can pass a check on those rows because the fit
made its independences true there, so the check needs rows the graph was not learned from. We learn
on the first 1,000 rows and check on the other 1,000.

In [3]:
alarm = load_dataset("alarm", n=2000, seed=1)
data, truth = alarm.data, alarm.graph
learn, held = data.iloc[:1000], data.iloc[1000:]
true_cpdag = to_cpdag(truth)
print(data.shape[1], "variables,", len(truth.oriented_edges()), "true edges")

37 variables, 46 true edges


## Check a graph

The true graph is the control: its independences hold, so the check should not reject it. The
result prints the verdict, the number of independences tested, and the uniformity statistic.

In [4]:
check = andrey.markov_check(truth, held)
print(check)

Not rejected: the p-values of the 620 independences the graph implies are consistent with uniform (Anderson-Darling p = 0.656 > 0.05); 5.6% have p < 0.05, about 5% expected. A pass does not show the edges are right: a graph with extra edges also passes.
  620 tests | Anderson-Darling A^2 = 0.592 | each test on 500 rows | indep_test fisherz


## A damaged graph is rejected

Remove the edge `CO -> BP` from the true graph. The graph now claims that `BP` is independent of
the causes of `CO`, such as `LVFAILURE`, given `TPR`, which the data contradict. `table()` lists
every tested independence with its p-value, so the smallest ones show where the graph is wrong:

In [5]:
def with_pair(graph, source, target, edge):
    """Return a copy of the DAG with `source -> target` added, or removed when not `edge`."""
    marks = graph.to_numpy().copy()
    i, j = graph.labels.index(source), graph.labels.index(target)
    marks[i, j], marks[j, i] = (1, 2) if edge else (0, 0)  # tail at source, arrowhead at target
    return GraphStructure.from_numpy(marks, kind="dag", labels=graph.labels)


damaged = with_pair(truth, "CO", "BP", edge=False)
check = andrey.markov_check(damaged, held)
print(check)
pd.DataFrame(check.table()).nsmallest(3, "p_value")

Rejected: the p-values of the 621 independences the graph implies are not uniform (Anderson-Darling p = 9.66e-07 <= 0.05); 8.5% have p < 0.05, about 5% expected.
  621 tests | Anderson-Darling A^2 = 48.4 | each test on 500 rows | indep_test fisherz


,x,y,given,p_value
185,BP,INSUFFANESTH,[TPR],0.0
188,BP,LVFAILURE,[TPR],0.0
197,BP,STROKEVOLUME,[TPR],0.0


## What a pass does not show

The check rejects a missing edge only when the data can see the dependence. `PULMEMBOLUS -> PAP`
is weak enough here that a graph without it still passes. A graph with an extra edge passes too,
because an extra edge removes an independence from the graph's claims and leaves the rest true.
So a pass says the data do not contradict the graph, not that its edges are right:

In [6]:
variants = {
    "without PULMEMBOLUS -> PAP": with_pair(truth, "PULMEMBOLUS", "PAP", edge=False),
    "plus HYPOVOLEMIA -> CVP": with_pair(truth, "HYPOVOLEMIA", "CVP", edge=True),
}
rows = []
for name, graph in variants.items():
    check = andrey.markov_check(graph, held)
    rows.append(
        {
            "graph": name,
            "edges": len(graph.oriented_edges()),
            "tests": check.n_tests,
            "uniformity p": round(check.uniformity_p_value, 3),
            "verdict": "not rejected" if check.passed else "rejected",
        }
    )
pd.DataFrame(rows)

,graph,edges,tests,uniformity p,verdict
0,without PULMEMBOLUS -> PAP,45,621,0.216,not rejected
1,plus HYPOVOLEMIA -> CVP,47,619,0.903,not rejected


## Choosing a graph: the sparsest one that passes

When the truth is unknown, Ramsey, Andrews, and Spirtes (2024) use the check to choose among the
graphs a method returns over a range of one setting: drop the graphs the data reject and keep the
sparsest of the rest, the simplest explanation the data allow. GES's `penalty_discount` multiplies
the BIC penalty, and a larger value gives fewer edges. Fit GES on the learn rows over five values
and check each fit on the held rows:

In [7]:
rows, fits = [], {}
for penalty in (0.5, 1, 2, 4, 8):
    fit = andrey.ges(learn, penalty_discount=penalty)
    check = andrey.markov_check(fit, held)
    fits[penalty] = fit
    rows.append(
        {
            "penalty_discount": penalty,
            "edges": len(fit.structure.oriented_edges()),
            "errors": shd(fit.structure, true_cpdag),
            "uniformity p": round(check.uniformity_p_value, 3),
            "verdict": "not rejected" if check.passed else "rejected",
        }
    )
sweep = pd.DataFrame(rows)
sweep

,penalty_discount,edges,errors,uniformity p,verdict
0,0.5,59,18,0.267,not rejected
1,1.0,52,9,0.675,not rejected
2,2.0,51,8,0.149,not rejected
3,4.0,50,15,0.247,not rejected
4,8.0,50,15,0.247,not rejected


The data reject none of the five graphs, so the rule picks the sparsest, 50 edges at
`penalty_discount` 4 (8 returns the same graph). It is not the most accurate: its 15 errors against
the true CPDAG are more than the 8 of the 51-edge graph at 2 and the 9 of the 52-edge graph at 1.
The check rules out graphs the data contradict, and among the rest fewer edges is a preference,
not a measure of accuracy.

In [8]:
passing = sweep[sweep.verdict == "not rejected"]
sparsest = passing.loc[passing.edges.idxmin(), "penalty_discount"]
print("sparsest graph that passes: penalty_discount", sparsest)
andrey.viz.draw(fits[sparsest].structure)

sparsest graph that passes: penalty_discount 4.0


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1207 763" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="1207" height="763" fill="#fafbfc"/><line x1="171.82" y1="356.50" x2="262.31" y2="356.50" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="712.24" y1="171.39" x2="810.18" y2="171.39" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="1137.95" y1="339.27" x2="712.24" y2="335.34" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="1147.95,339.36 1137.99,334.27 1137.90,344.27" fill="#3a4658" stroke-linejoin="round"/><line x1="1140.46" y1="329.02" x2="1072.52" y2="294.33" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="1149.37,333.57 1142.74,324.57 1138.19,333.47" fill="#3a4658" stroke-linejoin="round"/><path d="M 712.20 336.29 Q 885.10 350.50 1039.26 296.07" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="1048.69,292.74 1037.59,291.36 1040.92,300.79" fill="#3a4658" stroke-linejoin="round"/><line x1="712.14" y1="333.56" x2="810.29" y2="320.93" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="705.91" y1="346.38" x2="811.38" y2="522.93" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="816.51,531.51 815.68,520.36 807.09,525.49" fill="#3a4658" stroke-linejoin="round"/><line x1="287.39" y1="361.29" x2="364.03" y2="391.65" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="373.33,395.33 365.87,387.00 362.19,396.29" fill="#3a4658" stroke-linejoin="round"/><line x1="1052.05" y1="309.62" x2="952.15" y2="547.70" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="1055.92,300.40 1047.44,307.69 1056.66,311.56" fill="#3a4658" stroke-linejoin="round"/><line x1="1044.76" y1="272.08" x2="956.27" y2="182.76" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="1051.80,279.18 1048.31,268.56 1041.21,275.60" fill="#3a4658" stroke-linejoin="round"/><line x1="1042.69" y1="486.45" x2="957.44" y2="551.78" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="1050.63,480.37 1039.65,482.48 1045.73,490.42" fill="#3a4658" stroke-linejoin="round"/><line x1="1038.00" y1="470.96" x2="960.09" y2="465.86" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="1047.97,471.61 1038.32,465.97 1037.67,475.95" fill="#3a4658" stroke-linejoin="round"/><line x1="1042.36" y1="629.12" x2="957.62" y2="567.35" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="1050.44,635.01 1045.31,625.08 1039.42,633.16" fill="#3a4658" stroke-linejoin="round"/><line x1="1037.95" y1="642.89" x2="960.12" y2="643.61" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="1047.95,642.79 1037.90,637.89 1037.99,647.88" fill="#3a4658" stroke-linejoin="round"/><line x1="362.79" y1="687.25" x2="288.09" y2="700.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="372.63,685.44 361.89,682.33 363.70,692.17" fill="#3a4658" stroke-linejoin="round"/><line x1="393.47" y1="672.89" x2="471.16" y2="574.55" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="477.36,566.70 467.23,571.45 475.08,577.65" fill="#3a4658" stroke-linejoin="round"/><line x1="397.76" y1="679.02" x2="563.57" y2="624.34" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="573.07,621.21 562.01,619.59 565.14,629.09" fill="#3a4658" stroke-linejoin="round"/><line x1="59.05" y1="730.97" x2="145.82" y2="730.97" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="56.43" y1="723.14" x2="140.46" y2="659.72" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="148.44,653.69 137.45,655.73 143.47,663.71" fill="#3a4658" stroke-linejoin="round"/><line x1="929.14" y1="86.81" x2="833.34" y2="163.28" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><pol

## PC: no graph passes

The same sweep over PC's `alpha`, the level of its independence tests. PC returns a CPDAG, which
the check reads through a DAG of its class; a graph with no consistent DAG cannot be checked, and
`markov_check` raises `ValueError`:

In [9]:
rows = []
for alpha in (0.001, 0.01, 0.05, 0.1, 0.3):
    fit = andrey.pc(learn, alpha=alpha)
    row = {
        "alpha": alpha,
        "edges": len(fit.structure.oriented_edges()),
        "errors": shd(fit.structure, true_cpdag),
    }
    try:
        check = andrey.markov_check(fit, held)
        row["uniformity p"] = round(check.uniformity_p_value, 3)
        row["verdict"] = "not rejected" if check.passed else "rejected"
    except ValueError:
        row["uniformity p"], row["verdict"] = None, "no DAG to check"
    rows.append(row)
pd.DataFrame(rows)

,alpha,edges,errors,uniformity p,verdict
0,0.001,35,25,0.0,rejected
1,0.010,36,24,0.0,rejected
2,0.050,42,24,0.0,rejected
3,0.100,43,25,0.0,rejected
4,0.300,52,31,NaN,no DAG to check


At every `alpha` the data reject the graph PC returns, or the graph has no DAG to check, so there
is nothing to choose from. This is a finding about PC on these data and this sample size: PC's
graphs differ from the true CPDAG by many more errors than GES's, and the check agrees that they do
not fit the data. It does not say which edges to change; the smallest p-values in `table()` show
where to look. `andrey markov-check` runs the same check from the shell.